In [0]:
from pyspark.sql import functions as F

SOURCES = [
    {"path": "s3://your_bucket/walmart/customers/", "table": "customers"},
    {"path": "s3://your_bucket/walmart/employees/", "table": "employees"},
    {"path": "s3://your_bucket/walmart/order_items/", "table": "order_items"},
    {"path": "s3://your_bucket/walmart/orders/", "table": "orders"},
    {"path": "s3://your_bucket/walmart/products/", "table": "products"},
    {"path": "s3://your_bucket/walmart/stores/", "table": "stores"},
]

TARGET_SCHEMA = "walmart.bronze"
STATE_ROOT = "/Volumes/walmart/bronze/ingestion_state"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {TARGET_SCHEMA}.ingestion_state")

for source in SOURCES:
    table = source["table"]

    (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", f"{STATE_ROOT}/{table}/schema")
        .load(source["path"])
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .writeStream
        .format("delta")
        .option("mergeSchema", "true")
        .option("checkpointLocation", f"{STATE_ROOT}/{table}/checkpoint")
        .trigger(availableNow=True)
        .toTable(f"{TARGET_SCHEMA}.{table}")
    )